### RAG Pipeline - Data Ingestion to VectorDB Pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\acer\Documents\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
### Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"\n  ✓ Loaded {len(documents)} Pages")
        
        except Exception as e:
            print(f"  ✗ Error : {e}")

    print(f"Total documents loaded : {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: Artificial_Intelligence.pdf

  ✓ Loaded 5 Pages

Processing: Cloud_computing.pdf

  ✓ Loaded 6 Pages

Processing: Data_Science.pdf

  ✓ Loaded 16 Pages
Total documents loaded : 27


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-04-10T10:53:49+05:30', 'author': 'pc 4', 'moddate': '2023-04-10T10:53:49+05:30', 'source': '..\\data\\pdf\\Artificial_Intelligence.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'Artificial_Intelligence.pdf', 'file_type': 'pdf'}, page_content='© 2023 IJRTI | Volume 8, Issue 4 | ISSN: 2456-3315 \n  \nIJRTI2304061 International Journal for Research Trends and Innovation (www.ijrti.org) 356 \n \nRESEARCH PAPER ON ARTIFICIAL INTELLIGENCE & \nITS APPLICATIONS \n \n \nProf. Neha Saini \n \nAssistant Professor in Department of Computer Science & IT \nSDAM College Dinanagar \n \nABSTRACT- \n \nIt is the science and engineering of making intelligent machines, especially intelligent computer programs. It is related to \nthe similar task of using computers to understand human intelligence, but AI does not have to confine itself to methods that \nare biologically o

In [5]:
### Text Splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
         chunk_size = chunk_size,
         chunk_overlap = chunk_overlap,
         length_function = len,
         separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunks: ")
        print(f"\nContent: {split_docs[0].page_content[:200]}...")
        print(f"\nMetaData: {split_docs[0].metadata}")
        
    return split_docs


In [6]:
chunks = split_documents(all_pdf_documents)
chunks

Split 27 documents into 85 chunks

Example chunks: 

Content: © 2023 IJRTI | Volume 8, Issue 4 | ISSN: 2456-3315 
  
IJRTI2304061 International Journal for Research Trends and Innovation (www.ijrti.org) 356 
 
RESEARCH PAPER ON ARTIFICIAL INTELLIGENCE & 
ITS APP...

MetaData: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-04-10T10:53:49+05:30', 'author': 'pc 4', 'moddate': '2023-04-10T10:53:49+05:30', 'source': '..\\data\\pdf\\Artificial_Intelligence.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'Artificial_Intelligence.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-04-10T10:53:49+05:30', 'author': 'pc 4', 'moddate': '2023-04-10T10:53:49+05:30', 'source': '..\\data\\pdf\\Artificial_Intelligence.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'Artificial_Intelligence.pdf', 'file_type': 'pdf'}, page_content='© 2023 IJRTI | Volume 8, Issue 4 | ISSN: 2456-3315 \n  \nIJRTI2304061 International Journal for Research Trends and Innovation (www.ijrti.org) 356 \n \nRESEARCH PAPER ON ARTIFICIAL INTELLIGENCE & \nITS APPLICATIONS \n \n \nProf. Neha Saini \n \nAssistant Professor in Department of Computer Science & IT \nSDAM College Dinanagar \n \nABSTRACT- \n \nIt is the science and engineering of making intelligent machines, especially intelligent computer programs. It is related to \nthe similar task of using computers to understand human intelligence, but AI does not have to confine itself to methods that \nare biologically o

### Embedding and VectorDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

#Initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2472.14it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
      
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
            
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 170


### Convert the chunks into embeddings

In [10]:
texts = [doc.page_content for doc in chunks]

## Generate the Embeddings
embeddings = embedding_manager.generate_embeddings(texts)

## Store in the Vectore Database
vectorstore.add_documents(chunks, embeddings)


Generating embeddings for 85 texts...


Batches: 100%|██████████| 3/3 [00:05<00:00,  1.85s/it]


Generated embeddings with shape: (85, 384)
Adding 85 documents to vector store...
Successfully added 85 documents to vector store
Total documents in collection: 255


### Retriever Pipeline from VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
                
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

In [12]:
rag_retriever = RAGRetriever(vectorstore,embedding_manager)
rag_retriever

In [13]:
rag_retriever.retrieve("What is Ai")

Retrieving documents for query: 'What is Ai'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 20.74it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_7771f93e_0',
  'content': '© 2023 IJRTI | Volume 8, Issue 4 | ISSN: 2456-3315 \n  \nIJRTI2304061 International Journal for Research Trends and Innovation (www.ijrti.org) 356 \n \nRESEARCH PAPER ON ARTIFICIAL INTELLIGENCE & \nITS APPLICATIONS \n \n \nProf. Neha Saini \n \nAssistant Professor in Department of Computer Science & IT \nSDAM College Dinanagar \n \nABSTRACT- \n \nIt is the science and engineering of making intelligent machines, especially intelligent computer programs. It is related to \nthe similar task of using computers to understand human intelligence, but AI does not have to confine itself to methods that \nare biologically observable. While no consensual definition of Artificial Intelligence (AI) exists, AI is broadly characteriz ed \nas the study of computa tions that allow for perception, reason and action.  Today, the amount of data that is generated, by \nboth humans and machines, far outpaces humans’ ability to absorb, interpret, and make complex decis

In [14]:
rag_retriever.retrieve("Cloud Computing")

Retrieving documents for query: 'Cloud Computing'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.30it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_05a3e6d5_24',
  'content': 'technology that is transforming the way \nbusinesses operate, innovate, and sca le. By \ndelivering computing resources over the internet, \ncloud computing offers unparalleled flexibility, \ncost-efficiency, and accessibility. Enterprises of \nall sizes are increasingly adopting cloud services \nto optimize their operations, enhance collaboration, \nand gain a competitive edge. At its core, cloud \ncomputing allows businesses to move away from \nthe traditional model of maintaining and managing \nphysical IT infrastructure. Instead, they can \nleverage the resources provided by cloud service \nproviders, such as Amazon Web Services (AWS), \nMicrosoft Azure, and Google Cloud, to meet their \ncomputing needs on -demand. This shift not only \nreduces the overhead costs associated with \nmaintaining hardware and software but also opens \nup new opportunities for innovatio n and growth. \nIn this article, we will explore the critical aspects \nof cl

### Integration Vector Context Pipeline with LLM Output

In [15]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,
             model_name="llama-3.1-8b-instant",
             temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [16]:
q1 = rag_simple("What is Ai?",rag_retriever,llm,)
print(q1)

Retrieving documents for query: 'What is Ai?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.24it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


It is the science and engineering of making intelligent machines, especially intelligent computer programs.


In [17]:
q2 = rag_simple("What is Cloud Computing and it's types?", rag_retriever, llm)
print(q2)

Retrieving documents for query: 'What is Cloud Computing and it's types?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.72it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Cloud Computing is a technology that delivers computing resources over the internet, offering flexibility, cost-efficiency, and accessibility. It allows businesses to move away from traditional physical IT infrastructure and leverage resources provided by cloud service providers on-demand.

There are three main types of Cloud Computing:

1. **Public Cloud**: A public cloud is a multi-tenant cloud environment where resources are shared among multiple customers. Examples include Amazon Web Services (AWS), Microsoft Azure, and Google Cloud.
2. **Private Cloud**: A private cloud is a single-tenant cloud environment where resources are dedicated to a single organization. It can be hosted on-premises or off-premises.
3. **Hybrid Cloud**: A hybrid cloud is a combination of public and private clouds, allowing organizations to leverage the benefits of both environments.


### Enhanced RAG Pipeline features

In [18]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is Data Science", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is Data Science'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.06it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Data Science is the art of collecting and analyzing huge amounts of data from scientific fields and applications, using statistics or mathematics, to solve complex data problems.
Sources: [{'source': 'Data_Science.pdf', 'page': 7, 'score': 0.4483407139778137, 'preview': 'International Journal of Research \n(IJR) \np-ISSN: 2348-6848 \ne-ISSN: 2348-795X \nVol. 8 Issue 10 \nOctober 2021 \n \nCopyright © authors 2021 \n 59 \nFigure 7. Applications or domains of data science  \nIV. ROLE OF DATA SCIENTIST \nData science [5] continues to evolve as one of the promising and in -demand ...'}, {'source': 'Data_Science.pdf', 'page': 7, 'score': 0.4483407139778137, 'preview': 'International Journal of Research \n(IJR) \np-ISSN: 2348-6848 \ne-ISSN: 2348-795X \nVol. 8 Issue 10 \nOctober 2021 \n \nCopyright © authors 2021 \n 59 \nFigure 7. Applications or domains of data science  \nIV. ROLE OF DATA SCIENTIST \nData science [5] continues to evolve as one of the promising and in -demand ...'}, {